# Cluster Main Effects

LMM/GLMM 混合模型比较三个 cluster 在原创性、流畅性、高质量回答上的差异。


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import statsmodels.api as sm
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from pathlib import Path

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

cluster_name_map = {
    0: '广泛试探型',
    1: '高效采纳型',
    2: '审慎批判型',
}

In [ ]:
# 输入文件路径
output_dir = Path('output')
trial_with_cluster_file = output_dir / 'trial_with_cluster.csv'

data_dir = Path('../../../data/analysis')
performance_file = data_dir / 'performance' / 'performance.csv'

In [ ]:
# 工具函数

# ---- GLMM 两两比较（基于固定效应估计，在对数链接尺度上）----
# 说明：针对前面拟合的 PoissonBayesMixedGLM（fluency 与 above_median），
# 计算不同 cluster 水平之间的两两对比：估计差值（link），标准误，z / p，
# 以及对数链接尺度的 95% 置信区间与 IRR（exp(diff)）区间。
import patsy
from itertools import combinations
from statsmodels.stats.multitest import multipletests
import numpy as np
from scipy import stats as _sps
import pandas as _pd

def pairwise_glmm_contrasts(glmm, glmm_res, df, formula='C(cluster) + dat_score + ribs_total + cse_total + ai_attitude', cluster_col='cluster', covariates=None, model_label='model'):
    exog_names = list(glmm.exog_names)
    k_fe = len(exog_names)

    # 获取固定效应及其协方差：VB 有 fe_mean / fe_sd，MAP 有 params / cov_params
    if hasattr(glmm_res, 'fe_mean'):
        fe_mean = np.asarray(glmm_res.fe_mean).reshape(-1)
        fe_sd = np.asarray(glmm_res.fe_sd).reshape(-1)
        cov_fe = np.diag(fe_sd ** 2)
        cov_fe = _pd.DataFrame(cov_fe, index=exog_names, columns=exog_names)
    else:
        all_params = np.asarray(glmm_res.params).reshape(-1)
        fe_mean = all_params[:k_fe]
        try:
            cov = np.asarray(glmm_res.cov_params())
            cov_fe = _pd.DataFrame(cov, index=glmm.exog_names, columns=glmm.exog_names)
            cov_fe = cov_fe.reindex(index=exog_names, columns=exog_names).fillna(0.0)
        except Exception:
            cov_fe = _pd.DataFrame(np.nan, index=exog_names, columns=exog_names)

    # cluster 水平
    if isinstance(df[cluster_col].dtype, pd.CategoricalDtype):
        clusters = df[cluster_col].cat.categories
    else:
        clusters = sorted(df[cluster_col].dropna().unique())

    pred_df = _pd.DataFrame({cluster_col: clusters})
    # 填充协变量（默认使用样本均值），covariates 可以传入 dict 覆盖
    if covariates is None:
        covariates = {}
    for cov, val in covariates.items():
        if val is None and cov in df.columns:
            pred_df[cov] = df[cov].mean()
        else:
            pred_df[cov] = val

    # 若连续协变量未传且存在于 df 中，则设为均值
    for cov in ['dat_score', 'ribs_total', 'cse_total', 'ai_attitude']:
        if cov not in pred_df.columns and cov in df.columns:
            pred_df[cov] = df[cov].mean()

    design = patsy.dmatrix(formula, pred_df, return_type='dataframe')
    # 保证 design 列与 exog_names 对齐
    for col in exog_names:
        if col not in design.columns:
            design[col] = 0.0
    design = design[exog_names]

    results = []
    for i, j in combinations(range(len(clusters)), 2):
        xa = design.iloc[i].values
        xb = design.iloc[j].values
        contrast = xa - xb
        diff = float(contrast.dot(fe_mean))
        # 计算方差（若协方差包含 NaN 则返回 NaN）
        try:
            var = float(contrast.dot(cov_fe.values).dot(contrast))
        except Exception:
            var = np.nan
        se = np.sqrt(var) if not np.isnan(var) else np.nan
        zstat = diff / se if (se and not np.isnan(se)) else np.nan
        pval = 2 * _sps.norm.sf(abs(zstat)) if not np.isnan(zstat) else np.nan
        # 95% CI 在 link（对数）尺度上，再转换到 IRR（exp）尺度
        if not np.isnan(se):
            ci_lo = diff - 1.96 * se
            ci_hi = diff + 1.96 * se
            irr = np.exp(diff)
            irr_lo = np.exp(ci_lo)
            irr_hi = np.exp(ci_hi)
        else:
            ci_lo = ci_hi = irr = irr_lo = irr_hi = np.nan
        results.append({
            'group1': clusters[i],
            'group2': clusters[j],
            'est_link': diff,
            'se': se,
            'z': zstat,
            'p': pval,
            'ci_lo_link': ci_lo,
            'ci_hi_link': ci_hi,
            'irr': irr,
            'irr_lo': irr_lo,
            'irr_hi': irr_hi
        })

    res_df = _pd.DataFrame(results).sort_values('p')
    if len(res_df):
        res_df['p_bonf'] = multipletests(res_df['p'].values, method='bonferroni')[1]
        res_df['p_fdr'] = multipletests(res_df['p'].values, method='fdr_bh')[1]
    print(f'--- {model_label} 两两比较（按 p 排序）---')
    display(res_df.round(4))
    return res_df


In [ ]:
# 加载数据
df_cluster = pd.read_csv(trial_with_cluster_file)  # 包含 participant_id, trial_index, cluster 与行为特征
df_perf = pd.read_csv(performance_file)             # 包含 participant_id, item_name, trial_index, fluency, originality

# 以试次为单位合并聚类标签
df_merged = pd.merge(
    df_perf,
    df_cluster,
    on=['participant_id', 'trial_index'],
    how='left'
 )

print(df_merged['cluster'].value_counts(dropna=False).sort_index())
df_merged.describe()

In [ ]:
# 中文标签映射（供调节分析使用）
corr_labels_cn = {
    'trial_calls': 'AI调用次数',
    'first_ai_time': '首次调用时间',
    'pre_first_call_ideas': '首次调用前想法数',
    'pre_think_time': '平均调用前思考时间',
    'perspective_taking': '观点采纳率',
    'affected_by_ai': '受AI影响率',
    'bfi_extroversion': '外向性',
    'bfi_agreeableness': '宜人性',
    'bfi_conscientiousness': '尽责性',
    'bfi_neuroticism': '神经质',
    'bfi_openness': '开放性',
    'cse_total': '创造性自我效能',
    'ribs_total': 'RIBS总分',
    'ai_attitude': 'AI态度',
    'dat_score': 'DAT分数',
    'originality': '原创性',
    'fluency': '流畅性',
    'above_median': '高质量回答数',
    'above_median_ratio': '高质量回答率',
}


## 2. 不同AI调用模式组间的绩效差异比较

### 混合线性模型

In [ ]:
# 试次级 LMM：
# - 固定效应：cluster
# - 随机效应：participant_id（被试随机截距）+ item_name（题目随机截距）
# - 因变量：originality

# 读取试次绩效数据
performance_df = pd.read_csv(performance_file)

# 合并试次级 cluster 标签
performance_with_cluster = pd.merge(
    performance_df,
    df_cluster,
    on=['participant_id', 'trial_index'],
    how='left'
 )

# 仅保留有效 cluster，并清理缺失值
lmm_df = performance_with_cluster[performance_with_cluster['cluster'] >= 0].copy()
lmm_df = lmm_df.dropna(subset=['originality', 'cluster', 'dat_score', 'ribs_total', 'cse_total', 'ai_attitude', 'participant_id', 'item_name'])

# 设置类型，确保分类变量按类别处理
lmm_df['cluster'] = lmm_df['cluster'].astype('category')
lmm_df['participant_id'] = lmm_df['participant_id'].astype('category')
lmm_df['item_name'] = lmm_df['item_name'].astype('category')

lmm_df['cluster_name'] = lmm_df['cluster'].map(cluster_name_map)

print('LMM 数据规模:', lmm_df.shape)
print('被试数:', lmm_df['participant_id'].nunique())
print('物品数:', lmm_df['item_name'].nunique())
print('cluster 分布:\n', lmm_df['cluster'].value_counts().sort_index())

#### 原创性

In [ ]:
# 交叉随机截距模型：
# 固定效应：cluster + dat_score + ribs_total + cse_total + ai_attitude
# groups 指定 participant_id 随机截距；vc_formula 再加入 item_name 随机截距
lmm = sm.MixedLM.from_formula(
    formula='originality ~ C(cluster) + dat_score + ribs_total + cse_total + ai_attitude',
    groups='participant_id',
    re_formula='1',
    vc_formula={'item_name': '0 + C(item_name)'},
    data=lmm_df
)

# 使用 ML（reml=False）便于做固定效应模型比较
lmm_res = lmm.fit(reml=False, method='lbfgs', maxiter=2000)
print(lmm_res.summary())

# 固定效应项（含 cluster）的 Wald 检验
print('\n固定效应 Wald 检验:')
print(lmm_res.wald_test_terms(skip_single=False, scalar=True))

# 给出各 cluster 的模型预测均值（在同一被试与题目参照下）
pred_df = pd.DataFrame({'cluster': lmm_df['cluster'].cat.categories})
pred_df['participant_id'] = '__marginal__'  # 新分组水平 → 随机效应 = 0，获得边际均值
pred_df['item_name'] = '__marginal__'
pred_df['dat_score'] = lmm_df['dat_score'].mean()
pred_df['ribs_total'] = lmm_df['ribs_total'].mean()
pred_df['cse_total'] = lmm_df['cse_total'].mean()
pred_df['ai_attitude'] = lmm_df['ai_attitude'].mean()
pred_df['pred_originality'] = lmm_res.predict(pred_df)
print('\n各 cluster 的预测原创性:')
print(pred_df[['cluster', 'pred_originality']])


In [ ]:
# ---- LMM 基于固定效应的两两比较（基于估计边际均值 / LSM）----
# 原理：使用固定效应参数与其协方差矩阵，构建各 cluster 的设计向量，
# 计算组间差值的估计值、标准误、t 统计量与 p 值（并提供置信区间），
# 最后做多重比较校正（Bonferroni 或 FDR）。
import patsy
from itertools import combinations
from scipy import stats
from statsmodels.stats.multitest import multipletests

# 提取固定效应参数与协方差（确保以 DataFrame/Index 对齐）
fe_params = lmm_res.fe_params
cov_fe = lmm_res.cov_params()
if hasattr(cov_fe, 'loc'):
    cov_fe = cov_fe.loc[fe_params.index, fe_params.index]
else:
    import pandas as _pd
    cov_fe = _pd.DataFrame(cov_fe, index=fe_params.index, columns=fe_params.index)

# 准备用于预测的 DataFrame：把协变量设为均值，保留 cluster 的每个水平
clusters = lmm_df['cluster'].cat.categories
pred_df = pd.DataFrame({'cluster': clusters})
# 将连续协变量设置为样本均值
for cov in ['dat_score', 'ribs_total', 'cse_total', 'ai_attitude']:
    if cov in lmm_df.columns:
        pred_df[cov] = lmm_df[cov].mean()

# 使用 patsy 构建设计矩阵（包含截距），并对齐到固定效应参数的顺序
design = patsy.dmatrix('C(cluster) + dat_score + ribs_total + cse_total + ai_attitude', pred_df, return_type='dataframe')
# 有时 patsy 会产生列顺序与 fe_params 不同，按 fe_params.index 重新排序（若缺列则补 0）
for col in fe_params.index:
    if col not in design.columns:
        design[col] = 0.0
design = design[fe_params.index]

# 预测各水平的边际均值（fixed-effects 预测）
pred_means = design.dot(fe_params)

# 自由度：优先使用模型提供的 df_resid，否则按残差自由度近似
df_resid = getattr(lmm_res, 'df_resid', max(1, lmm_df.shape[0] - len(fe_params)))

# 两两对比：构造对比向量并计算差值、SE、t、p、CI
results = []
for i, j in combinations(range(len(clusters)), 2):
    xa = design.iloc[i].values
    xb = design.iloc[j].values
    contrast = xa - xb
    diff = float(contrast.dot(fe_params.values))
    var = float(contrast.dot(cov_fe.values).dot(contrast))
    se = (var ** 0.5) if var >= 0 else float('nan')
    tstat = diff / se if se != 0 else float('nan')
    pval = 2 * stats.t.sf(abs(tstat), df_resid) if not np.isnan(tstat) else float('nan')
    tcrit = stats.t.ppf(0.975, df_resid)
    ci_lo = diff - tcrit * se
    ci_hi = diff + tcrit * se
    results.append({
        'group1': clusters[i],
        'group2': clusters[j],
        'mean1': float(pred_means.iloc[i]),
        'mean2': float(pred_means.iloc[j]),
        'mean_diff': diff,
        'se': se,
        't': tstat,
        'p': pval,
        'ci_lo': ci_lo,
        'ci_hi': ci_hi
    })

res_df = pd.DataFrame(results).sort_values('p')
# 多重比较校正 (Bonferroni & FDR)
if len(res_df):
    res_df['p_bonf'] = multipletests(res_df['p'].values, method='bonferroni')[1]
    res_df['p_fdr'] = multipletests(res_df['p'].values, method='fdr_bh')[1]

print('基于 LMM 固定效应的两两比较（按 p 值排序）:')
display(res_df[['group1','group2','mean1','mean2','mean_diff','se','t','p','p_bonf','p_fdr','ci_lo','ci_hi']].round(4))


In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x='cluster_name', y='originality', data=lmm_df)
plt.title('不同AI调用模式的原创性比较') 
plt.xlabel('')
plt.ylabel('原创性')
plt.show()

#### 流畅性

In [ ]:
# fluency 的 GLMM：固定效应为 cluster，随机效应为 participant_id 和 item_name
# 双轨策略：优先 VB（便于不确定性解释），若不收敛则回退 MAP
from statsmodels.genmod.bayes_mixed_glm import PoissonBayesMixedGLM
import warnings

# 复用前面已经整理好的 lmm_df
fluent_df = lmm_df.dropna(subset=['fluency', 'cluster', 'participant_id', 'item_name', 'dat_score', 'ribs_total', 'cse_total', 'ai_attitude']).copy()
fluent_df['cluster'] = fluent_df['cluster'].astype('category')
fluent_df['participant_id'] = fluent_df['participant_id'].astype('category')
fluent_df['item_name'] = fluent_df['item_name'].astype('category')

print('fluency 描述:')
print(fluent_df['fluency'].describe())

fluent_glmm = PoissonBayesMixedGLM.from_formula(
    formula='fluency ~ C(cluster) + dat_score + ribs_total + cse_total + ai_attitude',
    vc_formulas={
        'participant_id': '0 + C(participant_id)',
        'item_name': '0 + C(item_name)',
    },
    data=fluent_df,
)

fit_mode = None
fluent_glmm_res = None

# 先尝试 VB
with warnings.catch_warnings(record=True) as wlist:
    warnings.simplefilter('always')
    try:
        vb_res = fluent_glmm.fit_vb(
            fit_method='BFGS',
            minim_opts={'maxiter': 20000, 'gtol': 1e-5, 'disp': False},
            scale_fe=True,
        )
        vb_nonconverged = any('did not converge' in str(w.message).lower() for w in wlist)
        if vb_nonconverged:
            print('检测到 VB 未收敛，自动回退到 MAP。')
        else:
            fluent_glmm_res = vb_res
            fit_mode = 'VB'
    except Exception as e:
        print(f'VB 拟合失败，自动回退到 MAP。原因: {e}')

# 回退 MAP
if fluent_glmm_res is None:
    fluent_glmm_res = fluent_glmm.fit_map(method='BFGS', minim_opts={'maxiter': 20000, 'gtol': 1e-5, 'disp': False})
    fit_mode = 'MAP'

print(f'\n当前采用的拟合方式: {fit_mode}')
print(fluent_glmm_res.summary())

# 统一构造固定效应结果表
fe_names = list(fluent_glmm.exog_names)
k_fe = len(fe_names)

if fit_mode == 'VB':
    fe_mean = np.asarray(fluent_glmm_res.fe_mean).reshape(-1)
    fe_sd = np.asarray(fluent_glmm_res.fe_sd).reshape(-1)
    interval_label = '近似 95% 后验区间（VB）'
else:
    # MAP 采用点估计 + Hessian 近似标准误，区间为正态近似 CI
    all_params = np.asarray(fluent_glmm_res.params).reshape(-1)
    fe_mean = all_params[:k_fe]
    try:
        cov = np.asarray(fluent_glmm_res.cov_params())
        fe_sd = np.sqrt(np.diag(cov)[:k_fe])
    except Exception:
        fe_sd = np.full(k_fe, np.nan)
    interval_label = '近似 95% 置信区间（MAP, Hessian）'

z = 1.96
ci_lower = fe_mean - z * fe_sd
ci_upper = fe_mean + z * fe_sd

fe_df = pd.DataFrame({
    'term': fe_names,
    'coef': fe_mean,
    'sd': fe_sd,
    'ci_lower': ci_lower,
    'ci_upper': ci_upper,
})

# Poisson 链接函数下，exp(coef) 是率比（IRR）
fe_df['irr'] = np.exp(fe_df['coef'])
fe_df['irr_ci_lower'] = np.exp(fe_df['ci_lower'])
fe_df['irr_ci_upper'] = np.exp(fe_df['ci_upper'])

# 显著性直观标记：系数区间不含 0（等价于 IRR 区间不含 1）
fe_df['sig_95'] = (fe_df['ci_lower'] > 0) | (fe_df['ci_upper'] < 0)
fe_df['direction'] = np.where(fe_df['coef'] > 0, 'positive', 'negative')
fe_df['fit_mode'] = fit_mode

print(f'\n固定效应 {interval_label}:')
display(fe_df.round(4))

In [ ]:
try:
    if 'fluent_glmm' in globals() and 'fluent_glmm_res' in globals():
        print('\nFluency GLMM 两两比较（cluster）:')
        pairwise_glmm_contrasts(fluent_glmm, fluent_glmm_res, fluent_df, formula='C(cluster) + dat_score + ribs_total + cse_total + ai_attitude', cluster_col='cluster', covariates={'dat_score': fluent_df['dat_score'].mean() if 'dat_score' in fluent_df.columns else None, 'ribs_total': fluent_df['ribs_total'].mean() if 'ribs_total' in fluent_df.columns else None, 'cse_total': fluent_df['cse_total'].mean() if 'cse_total' in fluent_df.columns else None, 'ai_attitude': fluent_df['ai_attitude'].mean() if 'ai_attitude' in fluent_df.columns else None}, model_label='fluency_glmm')
except Exception as e:
    print('fluency 两两比较失败:', e)

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x='cluster_name', y='fluency', data=fluent_df)
plt.title('不同AI调用模式的流畅性比较')
plt.xlabel('')
plt.ylabel('流畅性')
plt.show()

#### 高质量回答数量

In [ ]:
# 对 above_median 指标进行 GLMM（泊松）分析，固定效应为 cluster，随机效应为 participant_id 和 item_name
from statsmodels.genmod.bayes_mixed_glm import PoissonBayesMixedGLM
import warnings

# 准备数据：复用之前的 lmm_df（已含 cluster, participant_id, item_name, above_median 等）
ab_df = lmm_df.copy()
# 假定前面步骤已生成离散计数变量 above_median（范围 0~24），直接去除缺失值并设置分类变量
ab_df = ab_df.dropna(subset=['above_median', 'cluster', 'participant_id', 'item_name', 'dat_score', 'ribs_total', 'cse_total', 'ai_attitude']).copy()
ab_df['cluster'] = ab_df['cluster'].astype('category')
ab_df['participant_id'] = ab_df['participant_id'].astype('category')
ab_df['item_name'] = ab_df['item_name'].astype('category')

print('above_median 描述:')
print(ab_df['above_median'].describe())

# 构建模型：固定效应为 cluster，随机效应为 participant_id 和 item_name（泊松链接）
ab_glmm = PoissonBayesMixedGLM.from_formula(
    formula='above_median ~ C(cluster) + dat_score + ribs_total + cse_total + ai_attitude',
    vc_formulas={
        'participant_id': '0 + C(participant_id)',
        'item_name': '0 + C(item_name)',
    },
    data=ab_df,
 )

fit_mode = None
ab_glmm_res = None

# 先尝试 VB，若不收敛则回退 MAP（与 fluency 部分一致的双轨策略）
with warnings.catch_warnings(record=True) as wlist:
    warnings.simplefilter('always')
    try:
        vb_res = ab_glmm.fit_vb(
            fit_method='BFGS',
            minim_opts={'maxiter': 20000, 'gtol': 1e-5, 'disp': False},
            scale_fe=True,
        )
        vb_nonconverged = any('did not converge' in str(w.message).lower() for w in wlist)
        if vb_nonconverged:
            print('检测到 VB 未收敛，自动回退到 MAP。')
        else:
            ab_glmm_res = vb_res
            fit_mode = 'VB'
    except Exception as e:
        print(f'VB 拟合失败，自动回退到 MAP。原因: {e}')

# 回退 MAP
if ab_glmm_res is None:
    ab_glmm_res = ab_glmm.fit_map(method='BFGS', minim_opts={'maxiter': 20000, 'gtol': 1e-5, 'disp': False})
    fit_mode = 'MAP'

print(f'\n当前采用的拟合方式: {fit_mode}')
print(ab_glmm_res.summary())

# 构造固定效应结果表（系数、近似置信/后验区间、率比 IRR）
fe_names = list(ab_glmm.exog_names)
k_fe = len(fe_names)

if fit_mode == 'VB':
    fe_mean = np.asarray(ab_glmm_res.fe_mean).reshape(-1)
    fe_sd = np.asarray(ab_glmm_res.fe_sd).reshape(-1)
    interval_label = '近似 95% 后验区间（VB）'
else:
    all_params = np.asarray(ab_glmm_res.params).reshape(-1)
    fe_mean = all_params[:k_fe]
    try:
        cov = np.asarray(ab_glmm_res.cov_params())
        fe_sd = np.sqrt(np.diag(cov)[:k_fe])
    except Exception:
        fe_sd = np.full(k_fe, np.nan)
    interval_label = '近似 95% 置信区间（MAP, Hessian）'

z = 1.96
ci_lower = fe_mean - z * fe_sd
ci_upper = fe_mean + z * fe_sd

fe_df = pd.DataFrame({
    'term': fe_names,
    'coef': fe_mean,
    'sd': fe_sd,
    'ci_lower': ci_lower,
    'ci_upper': ci_upper,
})

# 泊松链接下，exp(coef) 是率比（IRR）
fe_df['irr'] = np.exp(fe_df['coef'])
fe_df['irr_ci_lower'] = np.exp(fe_df['ci_lower'])
fe_df['irr_ci_upper'] = np.exp(fe_df['ci_upper'])
fe_df['sig_95'] = (fe_df['ci_lower'] > 0) | (fe_df['ci_upper'] < 0)
fe_df['direction'] = np.where(fe_df['coef'] > 0, 'positive', 'negative')
fe_df['fit_mode'] = fit_mode

print(f'\n固定效应 {interval_label}:')
display(fe_df.round(4))

In [ ]:
# 对 above_median 的 GLMM（若存在）做两两比较
try:
    if 'ab_glmm' in globals() and 'ab_glmm_res' in globals():
        print('\nAbove_median GLMM 两两比较（cluster）:')
        pairwise_glmm_contrasts(ab_glmm, ab_glmm_res, ab_df, formula='C(cluster) + dat_score + ribs_total + cse_total + ai_attitude', cluster_col='cluster', covariates={'dat_score': ab_df['dat_score'].mean() if 'dat_score' in ab_df.columns else None, 'ribs_total': ab_df['ribs_total'].mean() if 'ribs_total' in ab_df.columns else None, 'cse_total': ab_df['cse_total'].mean() if 'cse_total' in ab_df.columns else None, 'ai_attitude': ab_df['ai_attitude'].mean() if 'ai_attitude' in ab_df.columns else None}, model_label='above_median_glmm')
except Exception as e:
    print('above_median 两两比较失败:', e)


In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x='cluster_name', y='above_median', data=ab_df)
plt.title('不同AI调用模式的高质量回答数比较')
plt.xlabel('')
plt.ylabel('高质量回答数 ')
plt.show()

#### 高质量比率

In [ ]:
abr_df = lmm_df.dropna(subset=['above_median_ratio', 'cluster', 'dat_score', 'ribs_total', 'cse_total', 'ai_attitude', 'participant_id', 'item_name']).copy()

abr_df['cluster'] = abr_df['cluster'].astype('category')
abr_df['participant_id'] = abr_df['participant_id'].astype('category')
abr_df['item_name'] = abr_df['item_name'].astype('category')
print('above_median_ratio 描述:')
print(abr_df['above_median_ratio'].describe())

# 构建模型并拟合（与 originality LMM 保持一致的随机效应结构）
lmm_abratio = sm.MixedLM.from_formula(
    formula='above_median_ratio ~ C(cluster) + dat_score + ribs_total + cse_total + ai_attitude',
    groups='participant_id',
    re_formula='1',
    vc_formula={'item_name': '0 + C(item_name)'},
    data=abr_df,
)
lmm_abratio_res = lmm_abratio.fit(reml=False, method='lbfgs', maxiter=2000)
print(lmm_abratio_res.summary())

print('\n固定效应 Wald 检验:')
print(lmm_abratio_res.wald_test_terms(skip_single=False, scalar=True))

# 预测各 cluster 的模型均值（在示例被试与题目参照下）
pred_df = pd.DataFrame({'cluster': abr_df['cluster'].cat.categories})
pred_df['participant_id'] = '__marginal__'  # 新分组水平 → 随机效应 = 0，获得边际均值
pred_df['item_name'] = '__marginal__'
pred_df['dat_score'] = abr_df['dat_score'].mean()
pred_df['ribs_total'] = abr_df['ribs_total'].mean()
pred_df['cse_total'] = abr_df['cse_total'].mean()
pred_df['ai_attitude'] = abr_df['ai_attitude'].mean()
pred_df['pred_above_median_ratio'] = lmm_abratio_res.predict(pred_df)
print('\n各 cluster 的预测 above_median_ratio:')
print(pred_df[['cluster','pred_above_median_ratio']])

# 绘图：箱线图展示各 cluster 的比例分布
plt.figure(figsize=(6,4))
sns.boxplot(x='cluster_name', y='above_median_ratio', data=abr_df)
plt.title('不同AI调用模式的高质量回答比率比较')
plt.xlabel('')
plt.ylabel('高质量回答比率')
plt.show()

#### DAT分数

In [ ]:
# ANOVA 分析：检验不同 cluster 间的 dat_score 差异
anova_df = lmm_df.copy()
anova_df['cluster'] = anova_df['cluster'].astype('category')

# 一元方差分析（Type II）
model_dat = ols('dat_score ~ C(cluster)', data=anova_df).fit()
anova_table = sm.stats.anova_lm(model_dat, typ=2)
print('ANOVA (dat_score ~ cluster):')
print(anova_table)

# 事后比较：Tukey HSD
tukey = pairwise_tukeyhsd(endog=anova_df['dat_score'], groups=anova_df['cluster'], alpha=0.05)
print(tukey.summary())

# 可视化：箱线图
plt.figure(figsize=(6,4))
sns.boxplot(x='cluster_name', y='dat_score', data=anova_df)
plt.title('不同 AI 调用模式的 DAT 分数比较')
plt.xlabel('')
plt.ylabel('DAT 分数')
plt.show()

# 平均值
mean_dat = anova_df.groupby('cluster')['dat_score'].mean().reset_index()
print('各 cluster 的平均 DAT 分数:')
print(mean_dat.round(4))